# train.ipynb

Training notebook version of the clean CNN baseline pipeline.

## Notes


Clean training baseline for the fruit classification project.

Purpose:
- make the project look like engineering work instead of notebook-only coursework
- provide a reproducible baseline training pipeline
- keep the implementation compact and readable

This script does not reproduce every experiment from the notebook.
It gives a clear training path that matches the same project direction.


In [ ]:
"""
Clean training baseline for the fruit classification CNN.

- CLI-first (argparse) but safe to run in Jupyter
- Reproducible (seeds)
- GPU-safer (memory growth + modest defaults)
"""

from __future__ import annotations

import argparse
import json
import random
from pathlib import Path
from typing import Any

import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    BatchNormalization,
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    MaxPooling2D,
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator


# ---------- Utilities ----------

def setup_gpu_memory() -> None:
    """Enable dynamic GPU memory growth if a GPU is present."""
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"✅ Enabled memory growth on {len(gpus)} GPU(s)")
        except RuntimeError as e:
            print(f"⚠️ GPU memory setup warning: {e}")


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


def validate_data_dir(data_dir: Path) -> None:
    """Basic sanity checks on data directory."""
    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory not found: {data_dir}")

    class_dirs = [p for p in data_dir.iterdir() if p.is_dir()]
    if len(class_dirs) < 2:
        raise ValueError(f"Need at least 2 class folders in {data_dir}")

    total_images = 0
    for c in class_dirs:
        n = sum(1 for _ in c.glob("*.*"))
        if n == 0:
            raise ValueError(f"Class folder has no images: {c}")
        total_images += n

    if total_images < 20:
        raise ValueError(f"Too few images overall ({total_images}).")


# ---------- Model ----------

def build_model(
    input_shape: tuple[int, int, int],
    num_classes: int,
    learning_rate: float,
) -> tf.keras.Model:
    model = Sequential([
        tf.keras.Input(shape=input_shape),

        Conv2D(32, (3, 3), activation="relu"),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Conv2D(64, (3, 3), activation="relu"),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Conv2D(128, (3, 3), activation="relu"),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.5),
        Dense(num_classes, activation="softmax"),
    ])

    optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# ---------- Args ----------

def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Train a CNN fruit classifier.")
    parser.add_argument("--train_dir", type=Path, default=Path("../notebooks/data/train"))
    parser.add_argument("--img_size", type=int, default=100)
    parser.add_argument("--batch_size", type=int, default=8)     # safe default for notebook/GPU
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--learning_rate", type=float, default=5e-4)
    parser.add_argument("--validation_split", type=float, default=0.2)
    parser.add_argument("--seed", type=int, default=42)

    parser.add_argument("--output_model", type=Path, default=Path("model/fruit_classifier.keras"))
    parser.add_argument("--output_best_model", type=Path, default=Path("model/fruit_classifier_best.keras"))
    parser.add_argument("--output_history", type=Path, default=Path("model/training_history.json"))
    parser.add_argument("--output_classes", type=Path, default=Path("model/class_indices.json"))

    if argv is None:
        # Notebook-safe: ignore Jupyter's own -f kernel args
        args, _ = parser.parse_known_args()
    else:
        args = parser.parse_args(argv)
    return args


# ---------- Main training ----------

def main(args: argparse.Namespace | None = None) -> None:
    if args is None:
        args = parse_args()

    print("🚀 Fruit CNN Training")
    print(f"Config: {vars(args)}")

    setup_gpu_memory()
    set_seed(args.seed)

    # Ensure output folders exist
    args.output_model.parent.mkdir(parents=True, exist_ok=True)
    args.output_best_model.parent.mkdir(parents=True, exist_ok=True)
    args.output_history.parent.mkdir(parents=True, exist_ok=True)
    args.output_classes.parent.mkdir(parents=True, exist_ok=True)

    # Data checks
    try:
        validate_data_dir(args.train_dir)
    except Exception as e:
        print(f"❌ Data validation failed: {e}")
        return

    # Data generators
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255.0,
        validation_split=args.validation_split,
        rotation_range=20,
        zoom_range=0.2,
        shear_range=0.15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        brightness_range=(0.8, 1.2),
        horizontal_flip=False,
        fill_mode="nearest",
    )

    val_datagen = ImageDataGenerator(
        rescale=1.0 / 255.0,
        validation_split=args.validation_split,
    )

    train_gen = train_datagen.flow_from_directory(
        str(args.train_dir),
        target_size=(args.img_size, args.img_size),
        batch_size=args.batch_size,
        class_mode="categorical",
        subset="training",
        shuffle=True,
        seed=args.seed,
    )

    val_gen = val_datagen.flow_from_directory(
        str(args.train_dir),
        target_size=(args.img_size, args.img_size),
        batch_size=args.batch_size,
        class_mode="categorical",
        subset="validation",
        shuffle=False,
        seed=args.seed,
    )

    print(f"✅ Loaded {train_gen.samples} train, {val_gen.samples} val samples")
    print(f"Classes: {list(train_gen.class_indices.keys())}")

    # Model + callbacks
    model = build_model(
        input_shape=(args.img_size, args.img_size, 3),
        num_classes=train_gen.num_classes,
        learning_rate=args.learning_rate,
    )

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=args.output_best_model,
            monitor="val_loss",
            save_best_only=True,
            verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

    # Train
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=args.epochs,
        callbacks=callbacks,
        verbose=1,
    )

    # Save artifacts
    model.save(args.output_model)

    with args.output_history.open("w", encoding="utf-8") as f:
        json.dump(history.history, f, indent=2)

    with args.output_classes.open("w", encoding="utf-8") as f:
        json.dump(train_gen.class_indices, f, indent=2)

    print(f"✅ Saved final model: {args.output_model}")
    print(f"✅ Saved best model:  {args.output_best_model}")
    print(f"✅ Saved history:     {args.output_history}")
    print(f"✅ Saved classes:     {args.output_classes}")


# ---------- Notebook helper ----------

def notebook_train(**overrides: Any) -> None:
    """
    Convenience wrapper for Jupyter:
    notebook_train(batch_size=4, epochs=5)
    """
    args = parse_args([])
    for k, v in overrides.items():
        setattr(args, k, v)
    main(args)


if __name__ == "__main__":
    main()

🚀 Fruit CNN Training
Config: {'train_dir': WindowsPath('../notebooks/data/train'), 'img_size': 100, 'batch_size': 8, 'epochs': 20, 'learning_rate': 0.0005, 'validation_split': 0.2, 'seed': 42, 'output_model': WindowsPath('model/fruit_classifier.keras'), 'output_best_model': WindowsPath('model/fruit_classifier_best.keras'), 'output_history': WindowsPath('model/training_history.json'), 'output_classes': WindowsPath('model/class_indices.json')}
Found 4822 images belonging to 14 classes.
Found 1202 images belonging to 14 classes.
✅ Loaded 4822 train, 1202 val samples
Classes: ['Apple Granny Smith', 'Apricot', 'Avocado', 'Banana', 'Blueberry', 'Cactus fruit', 'Cherry', 'Corn', 'Kiwi', 'Mango', 'Orange', 'Pineapple', 'Strawberry', 'Watermelon']
